[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/relativity/closed_universe/waves.ipynb)

# Gravitational Waves of a Closed Universe

This example computes the gravitational waves of a closed universe, the round three-sphere, on the 600-cell's 600 regular tetrahedra, and watches them from inside through three small crystals. Regge calculus holds a curved space in the lengths of a mesh's edges: each tetrahedron is a piece of the space with those lengths, and the curvature gathers at the edges, as the deficit by which the angles of the tetrahedra round an edge fall short of a whole turn. A gravitational wave changes the lengths, and the deficits it causes pull them back. The slowest wave of the closed universe, its squash, stretches the whole sphere along one direction and squeezes it along another, and swings back. Three crystals of atoms on springs feel it differently: a soft one goes along with space, a stiff one keeps its shape, and one tuned to the swing rings with it.

Points of the sphere are modelled as antivectors, and a tetrahedron's ramps, the planes through its faces scaled to read one at the opposite corners, read a point's barycentric weights, `ramps & point`. With the squared lengths, the ramps' products give the metric of the cone over the tetrahedron from the sphere's centre, a map from points to planes, and through its inverse two ramps pair to the cosine of the angle between their faces: read so, each tetrahedron is a piece of the round sphere, and the round sphere's own chords leave no deficit on any mesh. The waves are the changes of the squared lengths that no motion of the vertices makes, kept so by one solve over the vertices. Stepped through time they make a smooth field of strain, read in the isoclinic frames: at every place a map from a step of the origin to the plane that reads its stretch, `strain(step) & step`, through which the crystals' atoms and the light roll.

In the notation of Regge calculus, the action of a curved three-space is $S = \sum_e \ell_e\,\epsilon_e$ over its edges' lengths and deficit angles, which by Schläfli's identity changes as $\delta S = \sum_e \epsilon_e\,\delta\ell_e$, and the linear waves of the squared lengths $q$ solve $M\ddot q = -K q$, with $K$ the Hessian of the potential and $M$ the discrete DeWitt metric, restricted to the transverse traceless changes.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
from collections.abc import Iterator
from functools import reduce
from operator import and_

import numpy as np
from numga import NumpyContext, stack
from numga.sparse import spdiag

from examples.util.animation import show_animation
from examples.util.topology import Topology
from examples.relativity.closed_universe import core, render, scenarios
from examples.relativity.closed_universe.core import ga

mv = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()                                                     # Scalar+
# Planes: the ramps across the tetrahedra, and those through the origin, the polar planes of its steps.
Plane = ga.gatype.vector()                                                      # Vector+
Through = Plane.derive.within("x y z")                                          # Vector+
# Points of the sphere and the origin's steps; the motors that carry them, and the rates that move and turn them.
Point = ga.gatype.antivector()                                                  # Antivector-
Step = Point.derive.containing("w")                                             # Antivector-
Motor = ga.gatype.rotor()                                                       # Rotor+
Rate = ga.gatype.bivector()                                                     # Bivector+
Translating = ga.gatype((Step, Through))                                        # Antivector- <- Vector+
# A strain, the plane that reads each step's stretch; a turning, the rate a step adds to a frame's turn; a
# stretching, the steps a map of them gives.
Strain = ga.gatype((Through, Step))                                             # Vector+ <- Antivector-
Turning = ga.gatype((Rate, Step))                                               # Bivector+ <- Antivector-
Stretching = ga.gatype((Step, Step))                                            # Antivector- <- Antivector-

# The origin's equator, the plane `w`, and the origin, its pole.
equator = mv.w                                                                  # [] Plane+
origin = equator.dual()                                                         # [] Point-
# The step the origin takes under the motion across a plane through it, a step by its value, cast so; its inverse
# gives each step its polar plane, and the meet of the equator with that plane is the rate that moves the origin along
# the step.
translating = origin.commutator(equator ^ Through).cast(Translating)            # [] Point- <- Plane+
polar = translating.inverse()                                                   # [] Plane+ <- Point-
moving = equator ^ polar(Step)                                                  # [] Rate+ <- Point-

## Isoclinic frames

In four dimensions the product of a rate with the extrinsic pseudoscalar `mv.xyzw` is the rate in the plane square to it, and as the pseudoscalar squares to one, it splits every rate into a half it keeps and a half it flips. Each half turns alike in two square planes, by the same angle in both, so that it moves every point of the sphere the same distance along a great circle. The quotient `place / origin` is the turn that carries the origin to a place in the plane of the great circle between them; its half that the pseudoscalar keeps, with one in the other half, still carries the origin there, turning alike in that plane and in the plane square to it: that motor is the isoclinic frame at the place. Carried along a step, the frames roll by `moving(step) + turning(step)`, where `turning`, the moving rate times the pseudoscalar, turns them in the plane square to it, and in them a heading keeps still along its great circle. The pseudoscalar is what picks out these turns, and with them a frame at every place that varies smoothly over the whole sphere.

In [ ]:
# The extrinsic pseudoscalar, whose product with a rate is the rate in the plane square to it; the rate that turns a
# frame in the plane square to each step's moving rate.
pseudoscalar = mv.xyzw                                                          # [] Antiscalar+
turning = moving * pseudoscalar                                                 # [] Rate+ <- Point-


def isoclinic(places: Point) -> Motor:
    """The isoclinic frames at places on the sphere, `Motor+`: the motors that carry the origin to each by turning
    alike in the plane of the great circle between them and in the plane square to it."""
    # The half of the quotient of the place by the origin that the pseudoscalar keeps, with one in the other half.
    return ((1 + pseudoscalar) * (places.normalized() / origin) + (1 - pseudoscalar)) / 2

## The 600-cell at rest

The 600-cell's 120 vertices lie on the sphere, placed by a turn, and its edges' squared lengths at rest are those of the round sphere's chords between them. Read as a piece of the round sphere, a tetrahedron is the cone over it from the sphere's centre: the hyperplane through its corners takes the centre, and each pair of ramps reads the squared chord between their corners, so that the squared lengths make the cone's metric. Through the inverse metric, the ramps of the two corners off an edge pair to the cosine of the angle between the faces that meet there, and a whole turn less those angles round the edge is its deficit. The deficits drive the lengths: the potential's slope along each squared length is the deficit, scaled by how the arc changes with the squared chord.

In [ ]:
cell_mesh, cell_vertices = scenarios.six_hundred_cell()
chords = cell_mesh.d[0] * cell_vertices                                         # Point-[edges]
rest = chords.dot(chords)                                                       # Scalar+[edges]
corners = cell_mesh.incidence[3, 0] * cell_vertices                             # [corners] Point-[cells]
# Each corner's ramp: the plane through the face across from it, scaled to read one at the corner.
faces = reduce(and_, corners[cell_mesh.corners[3, 2]])                          # [corners] Plane-[cells]
ramps = faces / (faces & corners)                                               # [corners] Plane+[cells]
# The hyperplane through the corners, one on each, and each pair of ramps' symmetric product, which reads the squared
# chord between their corners: weighted by the squared lengths, the cone's metric from the centre.
radial = ramps.sum(axis=0)                                                      # Plane+[cells]
tails, heads = cell_mesh.corners[3, 1]
cone_basis = -(ramps[tails] * (ramps[heads] & Point) + ramps[heads] * (ramps[tails] & Point)) / 2   # [cell_edges] (Plane+ <- Point-)[cells]
cone_map = (spdiag(cone_basis) * cell_mesh.incidence[3, 1]).sum(axis=0)         # (Plane+ <- Point-)[cells <- edges]


def deficits(squared_lengths: Scalar) -> Scalar:
    """The deficit angle at each edge, `Scalar+[edges]`, of the tetrahedra read as pieces of the round sphere with
    these squared lengths: a whole turn less the angles between the faces round the edge."""
    cone = radial * (radial & Point) + cone_map * squared_lengths               # (Plane+ <- Point-)[cells]
    pairings = ramps[:, None] & cone.batch().inverse().field()(ramps[None, :])   # [corners, corners] Scalar+[cells]
    # The faces at an edge are across from the two corners off it, the opposite edge's: in the tetrahedron's own order
    # of its edges, the order reversed.
    off_edges = pairings[tails[::-1], heads[::-1]]                              # [cell_edges] Scalar+[cells]
    sides = pairings[tails[::-1], tails[::-1]] * pairings[heads[::-1], heads[::-1]]   # [cell_edges] Scalar+[cells]
    angles = (-off_edges / sides.square_root()).arccos()                        # [cell_edges] Scalar+[cells]
    return 2 * np.pi - (~cell_mesh.incidence[3, 1] * angles).sum(axis=-1)


def slopes(squared_lengths: Scalar) -> Scalar:
    """The potential's slope along each squared length, `Scalar+[edges]`: the deficit, times how the arc changes with
    the squared chord, a half over the chord and the cosine of half the arc."""
    return -deficits(squared_lengths) / (squared_lengths * (1 - squared_lengths / 4)).square_root()

## Masses and the waves

The lengths' motion has a kinetic term of its own, relativity's on a flat tetrahedron: a quarter of its volume times the pairings, through its inverse metric, of the corners across two edges, both ways. The flat tetrahedron's metric comes from the same products, of its corner gradients, the ramps less a quarter of the hyperplane. The stiffness is the slopes' change with the lengths, found by nudging them. A move of the vertices changes each edge's squared length by twice the chord's reading of the move; such changes are no waves, and the waves are what is left once the change nearest by the masses that the moves make is taken away, a solve over the vertices.

In [ ]:
# The flat tetrahedra: the corner gradients, their products, and each one's metric and volume from the squared lengths.
gradients = ramps - radial / 4                                                  # [corners] Plane+[cells]
flat_basis = -(gradients[tails] * (gradients[heads] & Point) + gradients[heads] * (gradients[tails] & Point)) / 2   # [cell_edges] (Plane+ <- Point-)[cells]
flat_map = (spdiag(flat_basis) * cell_mesh.incidence[3, 1]).sum(axis=0)         # (Plane+ <- Point-)[cells <- edges]
flat = radial * (radial & Point) + flat_map * rest                              # (Plane+ <- Point-)[cells]
spans = corners[1:] - corners[0]                                                # [3] Point-[cells]
volumes = ((flat(spans[0]) ^ flat(spans[1]) ^ flat(spans[2]) ^ radial) & (corners[0] & corners[1] & corners[2] & corners[3])).abs().square_root() / 6   # Scalar+[cells]
pairings = gradients[:, None] & flat.batch().inverse().field()(gradients[None, :])   # [corners, corners] Scalar+[cells]
# Relativity's kinetic term between the rates of two edges' squared lengths, assembled over the tetrahedra.
local = volumes / 4 * (pairings[tails[:, None], tails[None, :]] * pairings[heads[:, None], heads[None, :]]
                       + pairings[tails[:, None], heads[None, :]] * pairings[heads[:, None], tails[None, :]])   # [cell_edges, cell_edges] Scalar+[cells]
edges = cell_mesh.incidence[3, 1]                                               # [cell_edges] numbers[cells <- edges]
masses = (~edges[:, None] * spdiag(local) * edges[None, :]).sum(axis=0).sum(axis=0) * Scalar   # (Scalar+ <- Scalar+)[edges <- edges]
# The stiffness found by nudging each squared length by this much.
nudge = 1e-6
stiffness = core.jacobian(slopes, rest, nudge, cell_mesh)                       # (Scalar+ <- Scalar+)[edges <- edges]
moves = spdiag(2 * chords.dot(Point)) * cell_mesh.d[0]                          # (Scalar+ <- Point-)[edges <- vertices]
# The moves of the whole sphere change no length: this share of the largest move's own square settles them at nothing.
settling = 1e-10
reach = moves.adjoint()(masses(moves))                                          # (Point- <- Point-)[vertices <- vertices]
reach = reach + settling * core.largest(reach) * Point


def onto_waves(changes: Scalar) -> Scalar:
    """Changes of the squared lengths kept to the waves, `Scalar+[edges]`: less the change the vertex moves make
    that is nearest them by the masses."""
    return changes - moves(reach.solve(moves.adjoint()(masses(changes))))

## The squash

Read in the isoclinic frames, an edge runs along a step of the origin, its round step, and the squash lengthens the edges along `x` and shortens those along `y`, each by its step's share along the one less its share along the other, the same everywhere in the frames. Kept to the waves, its frequency squared is its stiffness over its masses: a fiftieth short of the continuum's eight, on edges a fifth of a half turn long.

In [ ]:
def round_steps(graph: Topology, places: Point) -> Point:
    """Each edge's round step, `Point-[edges]`, a step of the origin: its head carried back by its tail's isoclinic
    frame, less the origin it is beside."""
    tails, heads = graph.incidence[1, 0] * places                               # Point-[edges], Point-[edges]
    return (isoclinic(tails).inverse() >> heads).cast(Step)


steps = round_steps(cell_mesh, cell_vertices)                                   # Point-[edges]
# The squash's change of the squared lengths, per unit swing of space: twice, as a squared length changes twice as
# fast as the length.
squash = 2 * rest * ((mv.x & steps).squared() - (mv.y & steps).squared()) / steps.dot(steps)   # Scalar+[edges]
pattern = onto_waves(squash)                                                    # Scalar+[edges]
squash_squared = float(((pattern * stiffness(pattern)).sites.sum() / (pattern * masses(pattern)).sites.sum()).to_array())
print(f"the squash's frequency squared: {squash_squared:.3f}")

## Stepped through time

The waves step by leapfrog, the stiffness's push turned into the rates' change by the masses and kept to the waves. Started from the squash at its swing long before the first frame, so that the crystals settle into the swing and the light that reaches the eye has set out, they ring through one swing more, every step kept.

In [ ]:
# The swing of space, the steps to a swing, and the swings the crystals settle over before the frames.
wave_swing = 0.12
n_steps_per_swing = 40
n_settling_swings = 8

In [ ]:
def waves_stepped(strains: Scalar, rates: Scalar, step: float) -> Iterator[tuple[Scalar, Scalar]]:
    """The squared lengths' changes and their rates, `Scalar+[edges]` each, at every step of a leapfrog under the
    stiffness and the masses, kept to the waves: half a kick, a drift, and the other half kick."""
    while True:
        yield strains, rates
        rates = rates - (step / 2) * onto_waves(masses.solve(stiffness(strains)))
        strains = strains + step * rates
        rates = rates - (step / 2) * onto_waves(masses.solve(stiffness(strains)))


frequency = np.sqrt(squash_squared)
period = 2 * np.pi / frequency
step = period / n_steps_per_swing
# From the settling swings and half a turn more, the light's time across the sphere, to the end of the frames' swing.
first = -(n_settling_swings + 1) * period - np.pi
n_levels = int(np.ceil((period - first) / step)) + 2
strains = wave_swing * np.cos(frequency * first) * pattern                     # Scalar+[edges]
rates = -wave_swing * frequency * np.sin(frequency * first) * pattern           # Scalar+[edges]
levels = [state for _, state in zip(range(n_levels), waves_stepped(strains, rates, step))]

## A smooth field of strain

A change of the squared lengths changes each tetrahedron's cone metric by `cone_map * changes`; read on the steps that the isoclinic frame at the tetrahedron's middle carries out to it, and carried back, it is the tetrahedron's strain in the frames, a map from a step of the origin to the plane that reads its stretch. Averaged over the tetrahedra at each vertex, the strains make a field that blends across each tetrahedron by a place's barycentric weights, and between the stored moments by their nearness, and where the strain changes across a tetrahedron, half its curl turns the frames. Through the field, the frames along a step no longer roll as the round sphere's do: the strain stretches the step that moves the origin and turns the frame about it. That rate is the connection the atoms and the light roll along; the squeeze, half the strain's rate read back as a step, is how fast the slices of space stretch under them. This cell takes several seconds, most of it reading the strain at every stored moment.

In [ ]:
# The isoclinic frame at each tetrahedron's middle, and how many tetrahedra share each vertex.
cell_frames = isoclinic(cell_mesh.incidence[3, 0].mean(axis=0) * cell_vertices)   # Motor+[cells]
counts = (~cell_mesh.incidence[3, 0] * mv.scalar(np.ones((cell_mesh.n_elements[3], 1))).field()).sum(axis=0)   # Scalar+[vertices]


def strain_field(changes: Scalar) -> tuple[Strain, Turning]:
    """The strain of a change of the squared lengths, `Scalar+[edges]`, at each vertex, `Strain[vertices]`: the
    average of its tetrahedra's strains, each the change of the cone's metric read on the steps the frame at the
    tetrahedron's middle carries out, and carried back; and each tetrahedron's turn, half the curl of the strain
    across it, `Turning[cells]`."""
    carried = cell_frames >> Step                                               # (Point- <- Point-)[cells]
    strains = (cell_frames.inverse() >> (cone_map * changes)(carried)).cast(Strain)   # (Plane+ <- Point-)[cells]
    at_vertices = (~cell_mesh.incidence[3, 0] * strains).sum(axis=0) / counts   # (Plane+ <- Point-)[vertices]
    # Each corner's ramp read on the carried steps, and the strain at each corner: their wedge, summed over the
    # corners, is the strain's change across the tetrahedron turned into the plane it turns in.
    corner_planes = (cell_frames.inverse() >> ramps).cast(Through)              # [corners] Plane+[cells]
    corner_strains = cell_mesh.incidence[3, 0] * at_vertices                    # [corners] (Plane+ <- Point-)[cells]
    return at_vertices, (corner_planes ^ corner_strains).sum(axis=0) / 2


class Waves:
    """The stepped waves as a smooth field over the tetrahedra and over time, read in the isoclinic frames: at each
    stored moment, from `first_time` every `level_step`, the strain at the vertices, its rate and its trace, and each
    tetrahedron's turn."""

    def __init__(self, levels: list, first_time: float, level_step: float) -> None:
        self.ramps, self.first_time, self.level_step = ramps, first_time, level_step
        fields = [strain_field(changes) for changes, _ in levels]
        self.strains = stack([strain.batch() for strain, _ in fields])           # [levels, vertices] Plane+ <- Point-
        self.turns = stack([turn.batch() for _, turn in fields])                 # [levels, cells] Rate+ <- Point-
        self.strain_rates = stack([strain_field(rates)[0].batch() for _, rates in levels])   # [levels, vertices] Plane+ <- Point-
        self.traces = translating(self.strains).trace()                          # [levels, vertices] Scalar+

    def at(self, cells: np.ndarray, places: Point, time: float) -> tuple[Turning, Stretching]:
        """The connection at the places `[places] Point-` in the tetrahedra `cells`, at a time: the rate the frames
        roll by along each round step, `[places] Rate+ <- Point-`, and the squeeze of a step, half the strain's rate
        read back as a step, `[places] Point- <- Point-`."""
        level = (time - self.first_time) / self.level_step
        early, share = int(level), level - int(level)
        place_corners = cell_mesh.elements[3][cells].T                          # [corners, places]
        place_ramps = ramps.batch()[:, cells]                                   # [corners, places] Plane+
        weights = (place_ramps & places[None]) / (place_ramps.sum(axis=0) & places)   # [corners, places] Scalar+

        def blended(values):
            """A value at the vertices, `[levels, vertices]`, at the places and the time."""
            between = values[early] * (1 - share) + values[early + 1] * share   # [vertices]
            return (weights * between[place_corners]).sum(axis=0)               # [places]

        turn = self.turns[early, cells] * (1 - share) + self.turns[early + 1, cells] * share   # [places] Rate+ <- Point-
        # A round step, stretched by the strain, moves the origin; turned less twice the strain plus half its trace, it
        # turns the frame square to it; and the strain's change adds its turn.
        stretch = translating(blended(self.strains)(Step))                      # [places] Point- <- Point-
        local = moving(Step + stretch / 2) + turning(Step - 2 * stretch + Step * (blended(self.traces) / 2)) + turn(Step)
        return local, translating(blended(self.strain_rates)) / 2


waves = Waves(levels, first, step)

## Three crystals

Three little tetrahedra of atoms, bonded along their edges by springs and dashpots, sit side by side ahead of the eye. The squash drives a tetrahedron's swing at root two times the root of its springs' stiffness: the tuned crystal's stiffness puts that swing at the waves' frequency, the soft one's is a sixteenth of it and the stiff one's sixteen times it, all damped to a quarter of critical. An atom's velocity is a step of the origin in the frame at its place: it turns with the connection along the atom's motion, the squeeze slows it as space stretches, and the atom's bonds pull it along their proper separations, the steps the connection makes of their round steps. The soft crystal's bonds stretch with space, the stiff one's keep their length, and the tuned one's swing beyond the waves.

In [ ]:
# The crystals' stiffness against the tuned one's, their damping as a share of critical, and the time step of their
# atoms. The frames over one swing and a frame's duration in milliseconds; the eye's distance from the crystals, the
# time their light takes to arrive.
tunings = np.array([1 / 16, 1, 16])                                             # [crystals]
critical_share = 0.25
crystal_step = 0.01
n_frames = 75
duration_ms = 40
crystal_distance = scenarios.crystal_distance

In [ ]:
pick = core.picking(cell_mesh, cell_vertices)


def massive(bonds: Topology, rest: Scalar, stiffness: Scalar, damping: Scalar, starts: Point, start: float,
            step: float, times: np.ndarray) -> Iterator[tuple[Point, Scalar]]:
    """Atoms of unit mass rolled through the waves from rest at `starts`, `Point-[atoms]`, at time `start`, held by
    springs and dashpots along the edges of `bonds` of rest proper lengths `rest`, stiffness `stiffness` and damping
    `damping`, `Scalar+[bonds]` each, and stepped by `step` of time with symplectic Euler: their places and the bonds'
    proper strains, `Scalar+[bonds]`, at each of the `times`."""
    positions, lengths, time = starts.batch(), rest, start
    velocities = mv(Step, np.zeros((positions.shape[0], 3)))                    # [atoms] Point-
    tails = bonds.incidence[1, 0][0]                                            # numbers[bonds <- atoms]
    for target in times:
        while time < target - step / 2:
            local, squeeze = waves.at(pick(positions), positions, time)         # [atoms] Rate+ <- Point-, Point- <- Point-
            # Each bond's proper separation: the step the connection at its tail moves the origin by along its round
            # step.
            proper = origin.commutator((tails * local.field())(round_steps(bonds, positions.field()))).cast(Step)   # Point-[bonds]
            previous_lengths, lengths = lengths, proper.dot(proper).square_root()   # Scalar+[bonds], Scalar+[bonds]
            # Springs and dashpots along the proper separations, pulling a stretched bond's ends together.
            pulls = (stiffness * (lengths - rest) + damping * (lengths - previous_lengths) / step) / lengths   # Scalar+[bonds]
            forces = -(~bonds.d[0] * (pulls * proper)).batch()                  # [atoms] Point-
            # The step each round step moves the origin by: a velocity's round step is the one it moves the origin by.
            stepping = origin.commutator(local(Step)).cast(Stretching)          # [atoms] Point- <- Point-
            # The velocity turned by the connection along the motion, squeezed and pushed by the springs; the turn's
            # step part only, cast so.
            carried_velocities = velocities - velocities.commutator(local(stepping.solve(velocities))).cast(Step) * step
            velocities = carried_velocities - squeeze(carried_velocities) * step + forces * step   # [atoms] Point-
            # Rolled along the round step of its velocity, on its great circle from the frame at its place.
            rolls = isoclinic(positions) * (moving(stepping.solve(velocities)) * (-step / 2)).exp()   # [atoms] Motor+
            positions = rolls >> origin                                         # [atoms] Point-
            time += step
        yield positions.field(), lengths / rest - 1


# The crystals' tetrahedra of atoms, their bonds' stiffness and damping, and a bond at rest as long as its round step.
bonds, atoms = scenarios.crystals(scenarios.crystal_places())
stiffnesses = frequency ** 2 / 2 * tunings                                      # [crystals]
dampings = critical_share * np.sqrt(2 * stiffnesses)                            # [crystals]
n_bonds_each = bonds.n_elements[1] // len(tunings)
bond_stiffness = mv.scalar(np.repeat(stiffnesses, n_bonds_each)[:, None]).field()   # Scalar+[bonds]
bond_damping = mv.scalar(np.repeat(dampings, n_bonds_each)[:, None]).field()        # Scalar+[bonds]
bond_steps = round_steps(bonds, atoms)                                          # Point-[bonds]
bond_rest = bond_steps.dot(bond_steps).square_root()                            # Scalar+[bonds]
# The light from the crystals left them their distance before it reached the eye.
eye_times = np.arange(n_frames) * period / n_frames                             # [frames]
states = massive(bonds, bond_rest, bond_stiffness, bond_damping, atoms,
                 eye_times[0] - crystal_distance - n_settling_swings * period, crystal_step, eye_times - crystal_distance)

## The view from inside

The eye looks out through the waves at the crystals, the 600-cell's edges around it. Every ray rolls back from the eye along the same connection as the atoms, its heading turned and squeezed by the strain, so the struts sway as the squash swings. This cell takes about half a minute, most of it rolling the atoms and tracing the frames.

In [ ]:
show_animation(render.waves_view(cell_mesh, cell_vertices, waves, scenarios.eye, eye_times,
                                 scenarios.guardrails(cell_vertices, scenarios.guardrail_sides), bonds, atoms, states),
               "closed_universe_600_cell", duration_ms)

The 600-cell's twelve guardrail decagons, great circles side by side, are drawn each in its own colour, and every other edge as a thin ghost. Each crystal's bonds are coloured by their strain, blue squeezed and red stretched; from left to right the crystals are the soft, the tuned and the stiff one.

In [ ]:
# checks
# The round sphere's chords leave no deficit, and the squash is a fiftieth short of the continuum's eight.
assert (deficits(rest).abs() < 1e-12).all()
assert 7.5 < squash_squared < 8